# User Behavior and Dropout Analysis

This notebook examines engagement differences between retained and dropout users in a **10-row simulated dataset**. The goal is to demonstrate a transparent product-analytics workflow, not to make causal or predictive claims.

## Business question

Which engagement signals are associated with dropout in the available sample, and what should a product team measure next before making retention decisions?

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from analyze import load_and_validate, summarize

DATA_PATH = ROOT / 'data' / 'users_behavior.csv'
sns.set_theme(style='whitegrid')

## Load and validate the data

The reusable validation function checks required columns, missing values, duplicate user IDs, valid outcome categories, and non-negative numeric values.

In [ ]:
df = load_and_validate(DATA_PATH)
print(f'Rows: {len(df)} | Columns: {len(df.columns)}')
print(f'Missing values: {df.isna().sum().sum()}')
print(f'Duplicate user IDs: {df.user_id.duplicated().sum()}')
df.head()

## Data dictionary

| Column | Interpretation | Known limitation |
|---|---|---|
| `user_id` | Synthetic user identifier | None in the current sample |
| `age` | Age in years | Simulated |
| `time_on_platform` | Relative platform time | Unit is not specified |
| `clicks` | Click count | Observation window is not specified |
| `session_time` | Session duration in seconds | Assumed from the existing project context |
| `decision` | Yes/no decision | Business event and timing are not defined |
| `drop_out` | 0 = retained, 1 = dropout | Dropout rule is not documented |

## Descriptive comparison

The table below compares group averages. These values describe this sample only.

In [ ]:
group_summary = summarize(df)
group_summary

In [ ]:
plot_df = df.assign(status=df['drop_out'].map({0: 'Retained', 1: 'Dropout'}))
metrics = [
    ('time_on_platform', 'Time on platform'),
    ('clicks', 'Clicks'),
    ('session_time', 'Session time (seconds)'),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (column, label) in zip(axes, metrics):
    sns.barplot(
        data=plot_df, x='status', y=column, hue='status', errorbar=None,
        palette={'Retained': '#287271', 'Dropout': '#D65A4A'},
        legend=False, ax=ax
    )
    ax.set(xlabel='', ylabel=label, title=label)
fig.suptitle('Average engagement by dropout status', fontweight='bold')
fig.tight_layout()
plt.show()

## Leakage check

Before modeling dropout, we need to check whether any feature directly reveals the outcome.

In [ ]:
decision_dropout = pd.crosstab(df['decision'], df['drop_out'])
decision_dropout

Every `yes` decision corresponds to retention and every `no` decision corresponds to dropout. Because the timing and meaning of `decision` are undefined, it must be treated as a possible target-leakage variable rather than used as a predictor.

## Findings

- The five retained users averaged 7.2 units of platform time, 18.0 clicks, and 284 seconds per session.
- The five dropout users averaged 2.4 units of platform time, 5.6 clicks, and 77 seconds per session.
- Engagement is lower among dropout users in all three measures in this constructed sample.
- `decision` perfectly separates the outcome and presents a serious leakage risk.

These are descriptive associations, not evidence that engagement causes retention.

## Limitations and recommended next data

A predictive model is intentionally omitted because 10 observations cannot support reliable training or evaluation. The next dataset should contain more users, timestamps, cohort information, acquisition channel, device, a documented observation window, and a precise dropout definition. `decision` should be used only if it is known at prediction time and does not encode the future outcome.